In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("paper", font_scale=1.2)
sns.set_style("whitegrid")

In [ ]:
indir = "/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/metrics_new_donor_systema"

df_per_pert = pl.read_csv(
    os.path.join(indir, "systema_metrics_per_perturbation.csv"),
    infer_schema_length=10000,
    schema_overrides={"split_or_wandb": pl.Utf8},
).to_pandas()

print(f"Per-perturbation shape: {df_per_pert.shape}")
print(f"Methods: {sorted(df_per_pert['method'].unique())}")
print(f"Donors: {sorted(df_per_pert['donor'].unique())}")
print(f"Counts per method:")
print(df_per_pert.groupby('method').size())

In [ ]:
method_display = {
    "cellflow": "CellFlow",
    "mean_model_1": "Mean model 1",
    "mean_model_2": "Mean model 2",
    "identity": "Identity",
    "closest_embedding": "Closest embedding",
}

color_dict = {
    "CellFlow": "#B12F8C",
    "Mean model 1": "#8F97A8",
    "Mean model 2": "#566573",
    "Identity": "#BDBDBD",
    "Closest embedding": "#E0E0E0",
}

method_order = ["CellFlow", "Closest embedding", "Mean model 1", "Mean model 2", "Identity"]

df_per_pert["model"] = df_per_pert["method"].map(method_display)

# Average across seeds/splits per (donor, perturbation)
meta_cols = ["model", "donor", "perturbation"]
metric_cols = [
    "pearson_delta_all", "pearson_delta_top20",
    "mse_delta", "rmse_delta",
    "jaccard_top20_de", "centroid_accuracy",
]

df = df_per_pert.groupby(meta_cols)[metric_cols].mean().reset_index()
print(f"Averaged shape: {df.shape}")
print(f"Counts per model:")
print(df.groupby('model').size())

# Systema metrics - boxplots

In [ ]:
outdir = "/ictstr01/home/icb/dominik.klein/git_repos/ot_pert_new/fig_2/revision/systema"

systema_metrics = [
    ("pearson_delta_all", "Pearson delta (all genes)"),
    ("pearson_delta_top20", "Pearson delta (top 20 DE)"),
    ("mse_delta", "MSE (delta)"),
    ("jaccard_top20_de", "Jaccard top-20 DE"),
    ("centroid_accuracy", "Centroid accuracy"),
]

fig, axes = plt.subplots(1, len(systema_metrics), figsize=(4 * len(systema_metrics), 4))
for ax, (col, title) in zip(axes, systema_metrics):
    data = df.dropna(subset=[col])
    if data.empty:
        ax.set_title(f"{title}\n(no data)")
        continue
    available = [m for m in method_order if m in data["model"].unique()]
    sns.boxplot(
        data=data, x="model", y=col, order=available,
        palette=color_dict, ax=ax, showfliers=False,
    )
    sns.stripplot(
        data=data, x="model", y=col, order=available,
        color=".3", size=2, alpha=0.5, ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(os.path.join(outdir, "systema_boxplots.png"), dpi=150, bbox_inches="tight")
fig.savefig(os.path.join(outdir, "systema_boxplots.pdf"), bbox_inches="tight")
plt.show()

# Summary bar plot (mean +/- SE)

In [ ]:
summary_metrics = [
    ("pearson_delta_all", "Pearson delta (all genes)"),
    ("pearson_delta_top20", "Pearson delta (top 20 DE)"),
    ("mse_delta", "MSE (delta)"),
    ("jaccard_top20_de", "Jaccard top-20 DE"),
    ("centroid_accuracy", "Centroid accuracy"),
]

fig, axes = plt.subplots(1, len(summary_metrics), figsize=(4 * len(summary_metrics), 4))
for ax, (col, title) in zip(axes, summary_metrics):
    data = df.dropna(subset=[col])
    if data.empty:
        ax.set_title(f"{title}\n(no data)")
        continue
    available = [m for m in method_order if m in data["model"].unique()]
    sns.barplot(
        data=data, x="model", y=col, order=available,
        palette=color_dict, ax=ax, errorbar="se",
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
fig.savefig(os.path.join(outdir, "systema_summary.png"), dpi=150, bbox_inches="tight")
fig.savefig(os.path.join(outdir, "systema_summary.pdf"), bbox_inches="tight")
plt.show()

# Summary table (mean +/- std per model)

In [ ]:
summary_cols = [
    "pearson_delta_all", "pearson_delta_top20",
    "mse_delta", "jaccard_top20_de", "centroid_accuracy",
]

summary = df.groupby("model")[summary_cols].agg(["mean", "std"]).round(4)
summary.columns = [f"{col}_{stat}" for col, stat in summary.columns]
summary = summary.loc[[m for m in method_order if m in summary.index]]
print(summary.to_string())